# Softmax

**Goal:** Implement stable softmax and log-softmax from scratch in PyTorch;  
demonstrate numerical stability via max-subtraction; validate against `F.softmax` / `F.log_softmax`;  
show temperature scaling; verify the softmax Jacobian against `torch.autograd`.

Cross-links: `[[cross-entropy-nll]]` (cross-entropy consumes log-softmax output),  
`[[backpropagation]]` (softmax Jacobian propagated backward through logits),  
`[[logistic-regression]]` (two-class special case).

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # noqa
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Softmax — Math Recap

For a logit vector `z ∈ ℝᴷ`:

```
softmax(z)_i = exp(z_i) / Σⱼ exp(z_j)
```

**Shift invariance:** adding a constant `c` to every logit cancels in numerator and denominator:

```
softmax(z - c)_i = exp(z_i - c) / Σⱼ exp(z_j - c)
                 = exp(z_i) · exp(-c) / (Σⱼ exp(z_j) · exp(-c))
                 = softmax(z)_i
```

Setting `c = max(z)` keeps all shifted logits ≤ 0, preventing `exp` overflow.  
This is the **numerically stable form**.

## Part 1 — Softmax From Scratch (Numerically Stable)

We subtract the row-wise max before calling `exp`.  
The function works on batches of shape `(B, K)` as well as single vectors `(K,)`.

In [2]:
def softmax_stable(z: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Numerically stable softmax via max-subtraction.

    Args:
        z:   Logit tensor (any shape).
        dim: Dimension along which to compute softmax.

    Returns:
        Probability tensor with same shape as ``z``; values sum to 1 along ``dim``.
    """
    # Subtract max along dim for numerical stability (shift invariance)
    z_shifted = z - z.max(dim=dim, keepdim=True).values
    exp_z = torch.exp(z_shifted)
    return exp_z / exp_z.sum(dim=dim, keepdim=True)


# --- Quick sanity check: probabilities sum to 1 ---
torch.manual_seed(0)
logits_1d = torch.tensor([1.0, 2.0, 4.0], device=device)
p = softmax_stable(logits_1d)
print(f"logits:       {logits_1d.tolist()}")
print(f"softmax:      {[round(v, 4) for v in p.tolist()]}")
print(f"sum of probs: {p.sum().item():.6f}  (should be 1.0)")

logits:       [1.0, 2.0, 4.0]
softmax:      [0.042, 0.1142, 0.8438]
sum of probs: 1.000000  (should be 1.0)


### Validation: Compare with `F.softmax`

In [3]:
torch.manual_seed(42)
B, K = 16, 10
logits_batch = torch.randn(B, K, device=device)

p_scratch = softmax_stable(logits_batch, dim=-1)
p_torch   = F.softmax(logits_batch, dim=-1)

assert torch.allclose(p_scratch, p_torch, atol=1e-5), (
    f"Max deviation from F.softmax: {(p_scratch - p_torch).abs().max().item():.2e}"
)
print("PASS: softmax_stable matches F.softmax")
print(f"Max abs diff: {(p_scratch - p_torch).abs().max().item():.2e}")
print(f"Output shape: {p_scratch.shape}")

PASS: softmax_stable matches F.softmax
Max abs diff: 2.98e-08
Output shape: torch.Size([16, 10])


## Part 2 — Numerical Stability: Naive vs. Stable

For large logits, `exp(z)` overflows to `inf`.  
The naive formula then produces `inf / inf = nan`.  
The stable version (max-subtraction) handles arbitrarily large logits correctly.

In [4]:
def softmax_naive(z: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Naive softmax WITHOUT max-subtraction — overflows on large logits."""
    exp_z = torch.exp(z)
    return exp_z / exp_z.sum(dim=dim, keepdim=True)


# Large logits that cause overflow
# MPS may not support float64; use cpu for this numerical demonstration
large_logits = torch.tensor([1000.0, 1001.0, 1002.0])  # cpu, float32

p_naive  = softmax_naive(large_logits)
p_stable = softmax_stable(large_logits)

print("Naive softmax on large logits:")
print(f"  exp values: {torch.exp(large_logits).tolist()}")
print(f"  probabilities: {p_naive.tolist()}  <- NaN due to inf/inf")
print()
print("Stable softmax on large logits:")
shifted = large_logits - large_logits.max()
print(f"  shifted logits: {shifted.tolist()}")
print(f"  exp values:     {torch.exp(shifted).tolist()}")
print(f"  probabilities:  {[round(v, 4) for v in p_stable.tolist()]}  <- correct")
print()

assert any(v != v for v in p_naive.tolist()), "Expected NaN in naive softmax"   # nan != nan
assert not any(v != v for v in p_stable.tolist()), "Expected no NaN in stable softmax"
print("PASS: naive overflows to NaN; stable version computes correct probabilities")

Naive softmax on large logits:
  exp values: [inf, inf, inf]
  probabilities: [nan, nan, nan]  <- NaN due to inf/inf

Stable softmax on large logits:
  shifted logits: [-2.0, -1.0, 0.0]
  exp values:     [0.1353352814912796, 0.3678794503211975, 1.0]
  probabilities:  [0.09, 0.2447, 0.6652]  <- correct

PASS: naive overflows to NaN; stable version computes correct probabilities


## Part 3 — Log-Softmax From Scratch

`log(softmax(z)_i) = z_i - max(z) - log(Σⱼ exp(z_j - max(z)))`

Computing `log(softmax(z))` directly (i.e. `log` after softmax) loses precision when  
probabilities are very small. The closed form above is the numerically safe version —  
the one frameworks use internally.

Log-softmax is important because cross-entropy loss is `−y · log(p)`, and many frameworks  
accept log-probabilities directly (`F.nll_loss`), avoiding a redundant `log` call.  
See `[[cross-entropy-nll]]`.

In [5]:
def log_softmax_stable(z: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Numerically stable log-softmax.

    Uses the identity: log(softmax(z))_i = (z_i - max(z)) - log(Σⱼ exp(z_j - max(z)))

    Args:
        z:   Logit tensor (any shape).
        dim: Dimension along which to compute log-softmax.

    Returns:
        Log-probability tensor (all values ≤ 0) with same shape as ``z``.
    """
    z_shifted = z - z.max(dim=dim, keepdim=True).values
    log_sum_exp = torch.log(torch.exp(z_shifted).sum(dim=dim, keepdim=True))
    return z_shifted - log_sum_exp


# Validate
torch.manual_seed(7)
logits_test = torch.randn(8, 12, device=device)

lp_scratch = log_softmax_stable(logits_test, dim=-1)
lp_torch   = F.log_softmax(logits_test, dim=-1)

assert torch.allclose(lp_scratch, lp_torch, atol=1e-5), (
    f"Max deviation from F.log_softmax: {(lp_scratch - lp_torch).abs().max().item():.2e}"
)
print("PASS: log_softmax_stable matches F.log_softmax")
print(f"Max abs diff:     {(lp_scratch - lp_torch).abs().max().item():.2e}")
print(f"All values <= 0:  {(lp_scratch <= 0).all().item()}")
print(f"exp sums to 1:    {torch.allclose(lp_scratch.exp().sum(dim=-1), torch.ones(8, device=device))}")

PASS: log_softmax_stable matches F.log_softmax
Max abs diff:     0.00e+00


All values <= 0:  True
exp sums to 1:    True


## Part 4 — Temperature Scaling

Temperature `T` controls the sharpness of the softmax distribution:

```
softmax(z / T)
```

- `T < 1` (low temperature): distribution becomes **sharper** — one class dominates  
- `T = 1`: standard softmax  
- `T > 1` (high temperature): distribution becomes **flatter** — more uniform

In [6]:
def softmax_temperature(z: torch.Tensor, temperature: float, dim: int = -1) -> torch.Tensor:
    """Temperature-scaled softmax.

    Args:
        z:           Logit tensor.
        temperature: Controls sharpness (T > 0). Lower = sharper.
        dim:         Softmax dimension.

    Returns:
        Probability tensor summing to 1 along ``dim``.
    """
    if temperature <= 0:
        raise ValueError(f"temperature must be > 0, got {temperature}")
    return softmax_stable(z / temperature, dim=dim)


logits_temp = torch.tensor([2.0, 1.0, 0.0], device=device)
temperatures = [0.5, 1.0, 2.0]

print(f"Logits: {logits_temp.tolist()}")
print()
for T in temperatures:
    p = softmax_temperature(logits_temp, T)
    entropy = -(p * torch.log(p + 1e-12)).sum().item()
    label = "sharp" if T < 1 else ("flat" if T > 1 else "standard")
    print(f"T={T:.1f} ({label:8s}): {[round(v, 4) for v in p.tolist()]}  entropy={entropy:.4f}")

# Validate at T=1 this matches standard softmax
p_t1 = softmax_temperature(logits_temp, 1.0)
assert torch.allclose(p_t1, softmax_stable(logits_temp), atol=1e-6)
print()
print("PASS: temperature=1 matches standard softmax")

Logits: [2.0, 1.0, 0.0]

T=0.5 (sharp   ): [0.8668, 0.1173, 0.0159]  entropy=0.4411
T=1.0 (standard): [0.6652, 0.2447, 0.09]  entropy=0.8324
T=2.0 (flat    ): [0.5065, 0.3072, 0.1863]  entropy=1.0202

PASS: temperature=1 matches standard softmax


In [7]:
fig, ax = plt.subplots(figsize=(8, 3))
class_labels = [f"class {i}" for i in range(3)]
x = range(3)
width = 0.22

for idx, T in enumerate([0.25, 0.5, 1.0, 2.0, 4.0]):
    p = softmax_temperature(logits_temp, T).cpu().numpy()
    offset = (idx - 2) * width
    bars = ax.bar([xi + offset for xi in x], p, width, label=f"T={T}", alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(class_labels)
ax.set_ylabel("Probability")
ax.set_title("Softmax Distribution Under Temperature Scaling (logits=[2, 1, 0])")
ax.legend(fontsize=8, loc="upper right")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("softmax_temperature.png", dpi=80)
plt.close()
print("Plot saved to softmax_temperature.png")

Plot saved to softmax_temperature.png


## Part 5 — Softmax Jacobian and Gradient Verification

The Jacobian `J` of softmax has entries:

```
J_ij = d p_i / d z_j = p_i * (δ_ij - p_j)
```

where `δ_ij` is the Kronecker delta. In matrix form:

```
J = diag(p) - p · pᵀ
```

For a scalar loss `L` and upstream gradient `g = dL/dp` (shape `(K,)`),  
the vector-Jacobian product (VJP) `dL/dz = Jᵀ g` is what backprop computes.

We verify the manual Jacobian-vector product against `torch.autograd.functional.vjp`.

In [8]:
def softmax_jacobian(p: torch.Tensor) -> torch.Tensor:
    """Compute the full K×K Jacobian of softmax at probability vector p.

    J_ij = p_i * (delta_ij - p_j)
         = diag(p) - outer(p, p)

    Args:
        p: Probability vector of shape (K,), output of softmax.

    Returns:
        Jacobian matrix of shape (K, K).
    """
    # diag(p) - p @ p^T
    return torch.diag(p) - torch.outer(p, p)


torch.manual_seed(99)
K = 5
z_jac = torch.randn(K, device=device)
p_jac = softmax_stable(z_jac)

J_manual = softmax_jacobian(p_jac)
print(f"Jacobian shape: {J_manual.shape}")
print(f"Row sums (should be 0): {J_manual.sum(dim=1).tolist()}")

# Verify J is symmetric (softmax Jacobian is symmetric)
assert torch.allclose(J_manual, J_manual.T, atol=1e-6), "Jacobian should be symmetric"
print("PASS: Jacobian is symmetric")

Jacobian shape: torch.Size([5, 5])
Row sums (should be 0): [-2.2351741790771484e-08, -3.725290298461914e-09, 0.0, 0.0, -9.313225746154785e-10]
PASS: Jacobian is symmetric


In [9]:
# Gradient check: VJP against torch.autograd
# torch.autograd.functional.vjp on MPS can be finicky; fall back to cpu if needed
z_cpu = z_jac.detach().cpu()

def softmax_fn(z):
    return softmax_stable(z.to(device))

# Use autograd directly: z requires_grad, compute p, apply upstream g
z_ag = z_cpu.clone().to(device).requires_grad_(True)
p_ag = softmax_stable(z_ag)

# Random upstream gradient g = dL/dp
torch.manual_seed(55)
g = torch.randn(K, device=device)

# Backward pass: simulate L = g . p
loss_jac = (g * p_ag).sum()
loss_jac.backward()

dz_autograd = z_ag.grad.clone()   # dL/dz computed by autograd

# Manual VJP: dL/dz = J^T g  (J is symmetric so J^T = J)
dz_manual = J_manual @ g

assert torch.allclose(dz_autograd, dz_manual, atol=1e-5), (
    f"VJP mismatch: max diff = {(dz_autograd - dz_manual).abs().max().item():.2e}"
)
print("PASS: manual Jacobian VJP matches torch.autograd")
print(f"Max abs diff: {(dz_autograd - dz_manual).abs().max().item():.2e}")
print()
print("dL/dz (autograd):", [round(v, 5) for v in dz_autograd.cpu().tolist()])
print("dL/dz (manual):  ", [round(v, 5) for v in dz_manual.cpu().tolist()])

PASS: manual Jacobian VJP matches torch.autograd


Max abs diff: 7.45e-09

dL/dz (autograd): [-0.01506, -0.10795, 0.11836, 0.01303, -0.00839]
dL/dz (manual):   [-0.01506, -0.10795, 0.11836, 0.01303, -0.00839]


## Part 6 — Connection to Cross-Entropy (Softmax + CE Gradient)

For a one-hot target `y` and predictions `p = softmax(z)`, the cross-entropy loss is:

```
L = -Σᵢ yᵢ · log(pᵢ)
```

Its gradient with respect to the logits `z` simplifies to:

```
dL/dz = p - y
```

This is the famous "prediction minus target" result.  
Frameworks implement `CrossEntropyLoss` as a single numerically stable operation  
(`log_softmax` + `nll_loss`) rather than separate `softmax` then `log` — see `[[cross-entropy-nll]]`.

In [10]:
torch.manual_seed(11)
K_ce = 4

z_ce   = torch.randn(K_ce, device=device)
target = torch.tensor(2, device=device)          # ground-truth class index
y_onehot = F.one_hot(target, num_classes=K_ce).float()

# Manual CE loss = -log(p_correct_class) = -log_softmax[target]
lp = log_softmax_stable(z_ce)
loss_manual = -lp[target]

# PyTorch CE loss (expects raw logits)
loss_torch = F.cross_entropy(z_ce.unsqueeze(0), target.unsqueeze(0))

assert torch.allclose(loss_manual, loss_torch.squeeze(), atol=1e-5), (
    f"CE loss mismatch: {loss_manual.item():.6f} vs {loss_torch.item():.6f}"
)
print("PASS: manual CE loss matches F.cross_entropy")
print(f"CE loss: {loss_manual.item():.6f}")

# Gradient check: dL/dz = p - y
z_ce_ag = z_ce.detach().clone().requires_grad_(True)
loss_ag_ce = F.cross_entropy(z_ce_ag.unsqueeze(0), target.unsqueeze(0))
loss_ag_ce.backward()

p_ce = softmax_stable(z_ce)
dz_manual_ce = p_ce - y_onehot   # p - y

assert torch.allclose(z_ce_ag.grad, dz_manual_ce, atol=1e-5), (
    f"CE gradient mismatch: max diff = {(z_ce_ag.grad - dz_manual_ce).abs().max().item():.2e}"
)
print("PASS: dL/dz = p - y matches autograd for softmax + CE")
print(f"p:            {[round(v, 4) for v in p_ce.tolist()]}")
print(f"y (one-hot):  {y_onehot.tolist()}")
print(f"dL/dz = p-y:  {[round(v, 4) for v in dz_manual_ce.tolist()]}")

PASS: manual CE loss matches F.cross_entropy
CE loss: 2.012416
PASS: dL/dz = p - y matches autograd for softmax + CE
p:            [0.0175, 0.3731, 0.1337, 0.4757]
y (one-hot):  [0.0, 0.0, 1.0, 0.0]
dL/dz = p-y:  [0.0175, 0.3731, -0.8663, 0.4757]


## Idiomatic PyTorch

| Operation | Idiomatic form |
|---|---|
| Softmax | `F.softmax(logits, dim=-1)` |
| Log-softmax | `F.log_softmax(logits, dim=-1)` |
| Softmax + CE | `F.cross_entropy(logits, targets)` — single stable op |
| Temperature | `F.softmax(logits / T, dim=-1)` |
| Attention weights | `F.softmax(scores, dim=-1)` |

**Do not** pass probabilities to `F.cross_entropy` — it expects raw logits.  
The combined operation is more numerically stable and avoids computing `log(softmax(z))`  
as two separate steps.

In [11]:
# Idiomatic softmax
p_idem = F.softmax(logits_batch, dim=-1)
assert torch.allclose(p_idem, p_scratch, atol=1e-5)
print("PASS: F.softmax (idiomatic) matches from-scratch")

# Idiomatic log-softmax
lp_idem = F.log_softmax(logits_test, dim=-1)
assert torch.allclose(lp_idem, lp_scratch, atol=1e-5)
print("PASS: F.log_softmax (idiomatic) matches from-scratch")

# Idiomatic temperature scaling
p_temp_idem = F.softmax(logits_temp / 2.0, dim=-1)
p_temp_scratch = softmax_temperature(logits_temp, 2.0)
assert torch.allclose(p_temp_idem, p_temp_scratch, atol=1e-5)
print("PASS: temperature scaling (idiomatic) matches from-scratch")

PASS: F.softmax (idiomatic) matches from-scratch
PASS: F.log_softmax (idiomatic) matches from-scratch


PASS: temperature scaling (idiomatic) matches from-scratch


## Takeaways

1. **Max-subtraction is essential for numerical stability.**  
   `exp(1000)` overflows to `inf`; subtracting the max first keeps all shifted logits ≤ 0.

2. **Shift invariance is the key identity.**  
   `softmax(z) = softmax(z - c)` for any scalar `c` — this justifies the max trick.

3. **Log-softmax has its own stable formula.**  
   `log(softmax(z))` should not be computed as two separate steps — use `F.log_softmax`.

4. **The softmax Jacobian is `diag(p) - ppᵀ`.**  
   Row sums are zero (sum-to-one constraint). The VJP simplifies to `g - p·(g·p)`.

5. **With cross-entropy, `dL/dz = p - y`.**  
   The Jacobian complexity cancels out, giving the clean gradient that makes  
   gradient descent practical for classification. See `[[cross-entropy-nll]]`.

6. **Temperature controls sharpness.**  
   `T → 0` collapses to argmax; `T → ∞` approaches uniform.  
   Used in knowledge distillation, sampling, and policy smoothing.

Cross-links: `[[cross-entropy-nll]]` — `[[backpropagation]]` — `[[logistic-regression]]`